In [1]:
import pprint as pp

In [3]:
import pandas as pd
import re

# -------------------------------
# 1️⃣ Parse predicted table string
# -------------------------------
def parse_predicted_table(pred_table_str: str) -> pd.DataFrame:
    lines = [l.strip() for l in pred_table_str.strip().split("\n") if l.strip()]
    
    # Extract column line
    header_line = next((l for l in lines if l.startswith("<column>")), None)
    if not header_line:
        raise ValueError("No <column> header found in predicted table.")
    
    # Clean header and split by pipe
    header_clean = header_line.replace("<column>", "").strip()
    columns = [c.strip() for c in header_clean.split("|")]
    columns = [c for c in columns if c]  # Remove empty strings
    
    if not columns:
        raise ValueError("No columns extracted from header line.")
    
    # Extract rows
    rows = []
    for l in lines:
        if l.startswith("<row"):
            # Remove <row N> tag
            row_clean = re.sub(r"<row\s*\d+>\s*", "", l).strip()
            # Split by pipe
            parts = [p.strip() for p in row_clean.split("|")]
            
            # Only keep parts that match column count
            if len(parts) == len(columns):
                rows.append(parts)
            elif len(parts) > len(columns):
                # Truncate to match column count
                rows.append(parts[:len(columns)])
            else:
                # Pad with empty strings if fewer columns
                rows.append(parts + [""] * (len(columns) - len(parts)))
    
    if not rows:
        raise ValueError("No rows extracted from table.")
    
    df = pd.DataFrame(rows, columns=columns)
    return df


In [4]:
from collections import Counter
def get_correct_total_prediction_df(target_df, pred_df):
    """
    Compare two DataFrames and return sets of matched/mismatched
    rows, columns, and cells.
    """

    # Normalize as strings
    target_cells = [str(x).strip().lower() for x in target_df.values.flatten()]
    pred_cells = [str(x).strip().lower() for x in pred_df.values.flatten()]
    correct_cells = list((Counter(target_cells) & Counter(pred_cells)).elements())

    target_rows = [" | ".join(map(str, row)).strip().lower() for row in target_df.values.tolist()]
    pred_rows = [" | ".join(map(str, row)).strip().lower() for row in pred_df.values.tolist()]
    correct_rows = list((Counter(target_rows) & Counter(pred_rows)).elements())

    target_columns = [" | ".join(map(str, target_df[col])).strip().lower() for col in target_df.columns]
    pred_columns = [" | ".join(map(str, pred_df[col])).strip().lower() for col in pred_df.columns]
    correct_columns = list((Counter(target_columns) & Counter(pred_columns)).elements())

    return {
        "target_rows": target_rows,
        "target_columns": target_columns,
        "target_cells": target_cells,
        "pred_rows": pred_rows,
        "pred_columns": pred_columns,
        "pred_cells": pred_cells,
        "correct_rows": correct_rows,
        "correct_columns": correct_columns,
        "correct_cells": correct_cells
    }


In [5]:
import json
import pandas as pd
import os

def evaluate_results(result_file_path, failure_output_path=None):
    """
    Evaluates model responses globally across all queries.
    Computes total correct/total counts across dataset to derive
    global (micro) precision, recall, and F1 scores for cells, rows, and columns.
    
    If failure_output_path is not provided, auto-generates it in data/{language}/failure/ directory.
    Creates the failure directory if it doesn't exist.
    """

    with open(result_file_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    total_queries = 0
    parse_failures = 0
    failures_list = []

    # Global counters
    total_columns_in_dataset = 0
    total_rows_in_dataset = 0
    total_cells_in_dataset = 0
    total_correct_columns = 0
    total_correct_rows = 0
    total_correct_cells = 0
    total_predicted_rows_in_dataset = 0
    total_predicted_columns_in_dataset = 0
    total_predicted_cells_in_dataset = 0

    for table in results:
        table_name = table["table_name"]

        for query in table["queries"]:
            total_queries += 1
            try:
                # Extract model response
                model_response = query.get("model_response", {})
                if isinstance(model_response, str):
                    content = model_response
                else:
                    content = model_response.get("content", "")

                # Parse predicted table
                try:
                    pred_df = parse_predicted_table(content)
                except Exception as e:
                    parse_failures += 1
                    failures_list.append({
                        "table_name": table_name,
                        "question": query.get("question", ""),
                        "error": "1. "+str(e),
                        "model_response": content[:500]  # Store first 500 chars
                    })
                    continue

                # Get gold table
                gold_df = pd.DataFrame(query.get("result_df", []))

                # Compute detailed statistics
                stats = get_correct_total_prediction_df(gold_df, pred_df)

                total_columns_in_dataset += len(stats['target_columns'])
                total_rows_in_dataset += len(stats['target_rows'])
                total_cells_in_dataset += len(stats['target_cells'])
                total_correct_columns += len(stats['correct_columns'])
                total_correct_rows += len(stats['correct_rows'])
                total_correct_cells += len(stats['correct_cells'])
                total_predicted_rows_in_dataset += len(stats['pred_rows'])
                total_predicted_columns_in_dataset += len(stats['pred_columns'])
                total_predicted_cells_in_dataset += len(stats['pred_cells'])

            except Exception as e:
                parse_failures += 1
                failures_list.append({
                    "table_name": table_name,
                    "question": query.get("question", ""),
                    "error": "2. "+str(e),
                    "model_response": ""
                })
                continue

    # ---- Auto-generate failure path if not provided ----
    if failure_output_path is None and failures_list:
        # Extract language from path: data/{language}/...
        path_parts = result_file_path.split(os.sep)
        language = path_parts[1] if len(path_parts) > 1 else "unknown"
        
        # Get filename without extension
        filename = os.path.basename(result_file_path)
        base_name = os.path.splitext(filename)[0]
        
        # Build failure directory path
        failure_dir = os.path.join("data", language, "failure")
        failure_output_path = os.path.join(failure_dir, f"{base_name}_failures.json")

    # ---- Create failure directory if it doesn't exist ----
    if failure_output_path and failures_list:
        failure_dir = os.path.dirname(failure_output_path)
        os.makedirs(failure_dir, exist_ok=True)
        
        with open(failure_output_path, 'w', encoding='utf-8') as f:
            json.dump(failures_list, f, ensure_ascii=False, indent=2)
        # print(f"✓ Parse failures saved to {failure_output_path}")

    # ---- Compute global precision/recall/F1 ----
    def safe_div(a, b):
        return a / b if b > 0 else 0

    cell_precision = safe_div(total_correct_cells, total_predicted_cells_in_dataset)
    cell_recall = safe_div(total_correct_cells, total_cells_in_dataset)
    cell_f1 = safe_div(2 * cell_precision * cell_recall, (cell_precision + cell_recall))

    row_precision = safe_div(total_correct_rows, total_predicted_rows_in_dataset)
    row_recall = safe_div(total_correct_rows, total_rows_in_dataset)
    row_f1 = safe_div(2 * row_precision * row_recall, (row_precision + row_recall))

    column_precision = safe_div(total_correct_columns, total_predicted_columns_in_dataset)
    column_recall = safe_div(total_correct_columns, total_columns_in_dataset)
    column_f1 = safe_div(2 * column_precision * column_recall, (column_precision + column_recall))

    # Summary metrics
    global_metrics = {
        "total_queries": total_queries,
        "parse_failures": parse_failures,
        "success_rate": (total_queries - parse_failures) / total_queries if total_queries > 0 else 0,
        "cell_level": {
            "precision": cell_precision,
            "recall": cell_recall,
            "f1": cell_f1
        },
        "row_level": {
            "precision": row_precision,
            "recall": row_recall,
            "f1": row_f1
        },
        "column_level": {
            "precision": column_precision,
            "recall": column_recall,
            "f1": column_f1
        }
    }

    return global_metrics

In [6]:
import pandas as pd

# Dictionary to store all results
all_results = {}

languages = ["hindi", "telugu", "bengali"]
models = ["qwen3_14B", "qwen3_8B", "phi4"]
variants = [("", "with thinking"), ("_nothink", "without thinking")]

# Collect all results
for lang in languages:
    all_results[lang] = {}
    
    for model in models:
        all_results[lang][model] = {}
        
        for variant_suffix, variant_name in variants:
            if model == "phi4" and variant_suffix:  # phi4 doesn't have nothink variant
                continue
            
            if model == "phi4":
                file_path = f"data/{lang}/phi4_results_1.json"
            else:
                file_path = f"data/{lang}/{model}_results{variant_suffix}.json"
            
            try:
                metrics = evaluate_results(file_path)
                all_results[lang][model][variant_name] = metrics
            except FileNotFoundError:
                all_results[lang][model][variant_name] = None

# Print results in hierarchical format
for lang in languages:
    print(f"\n{'='*120}")
    print(f"  {lang.upper()}")
    print(f"{'='*120}")
    
    for model in models:
        print(f"\n  {model}")
        print(f"  {'-'*116}")
        
        for variant_name in ["with thinking", "without thinking"]:
            if variant_name not in all_results[lang][model]:
                continue
            
            metrics = all_results[lang][model][variant_name]
            
            if metrics is None:
                print(f"    {variant_name}: [File not found]")
                continue
            
            print(f"\n    {variant_name}:")
            print(f"      Queries: {metrics['total_queries']} | Failures: {metrics['parse_failures']} | Success Rate: {metrics['success_rate']*100:.2f}%")
            
            # Cell Level
            print(f"      Cell:")
            print(f"        Precision: {metrics['cell_level']['precision']:.4f}")
            print(f"        Recall:    {metrics['cell_level']['recall']:.4f}")
            print(f"        F1:        {metrics['cell_level']['f1']:.4f}")
            
            # Row Level
            print(f"      Row:")
            print(f"        Precision: {metrics['row_level']['precision']:.4f}")
            print(f"        Recall:    {metrics['row_level']['recall']:.4f}")
            print(f"        F1:        {metrics['row_level']['f1']:.4f}")
            
            # Column Level
            print(f"      Column:")
            print(f"        Precision: {metrics['column_level']['precision']:.4f}")
            print(f"        Recall:    {metrics['column_level']['recall']:.4f}")
            print(f"        F1:        {metrics['column_level']['f1']:.4f}")

print(f"\n\n{'='*120}")
print("  END OF REPORT")
print(f"{'='*120}\n")


  HINDI

  qwen3_14B
  --------------------------------------------------------------------------------------------------------------------

    with thinking:
      Queries: 1000 | Failures: 71 | Success Rate: 92.90%
      Cell:
        Precision: 0.5544
        Recall:    0.5141
        F1:        0.5335
      Row:
        Precision: 0.4643
        Recall:    0.4375
        F1:        0.4505
      Column:
        Precision: 0.5009
        Recall:    0.5160
        F1:        0.5083

    without thinking:
      Queries: 1000 | Failures: 3 | Success Rate: 99.70%
      Cell:
        Precision: 0.3372
        Recall:    0.5481
        F1:        0.4176
      Row:
        Precision: 0.2087
        Recall:    0.3374
        F1:        0.2579
      Column:
        Precision: 0.0519
        Recall:    0.0578
        F1:        0.0547

  qwen3_8B
  --------------------------------------------------------------------------------------------------------------------

    with thinking:
      Qu

In [7]:
import json
import os

def extract_parsing_cases(result_file_path):
    """
    Extract model outputs for cases where parsing is failing and passing
    """
    with open(result_file_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    passing_cases = []
    failing_cases = []
    
    for table in results:
        table_name = table["table_name"]
        
        for query in table["queries"]:
            question = query.get("question", "")
            model_response = query.get("model_response", {})
            
            if isinstance(model_response, str):
                content = model_response
            else:
                content = model_response.get("content", "")
            
            # Try to parse
            try:
                pred_df = parse_predicted_table(content)
                # Parsing successful
                passing_cases.append({
                    "table_name": table_name,
                    "question": question,
                    "model_response": content,
                    "status": "✓ PASS"
                })
            except Exception as e:
                # Parsing failed
                failing_cases.append({
                    "table_name": table_name,
                    "question": question,
                    "model_response": content,
                    "error": str(e),
                    "status": "✗ FAIL"
                })
    
    return passing_cases, failing_cases

# Set number of outputs to show
n = 3  # Change this to show more or fewer cases
output_file = "data/hindi/failure/qwen3_14B_parsing_analysis.txt"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Open file for writing
with open(output_file, 'w', encoding='utf-8') as f:
    # Extract cases for both variants
    f.write(f"\n{'='*140}\n")
    f.write(f"  QWEN3 14B - HINDI - PARSING ANALYSIS (showing top {n} cases)\n")
    f.write(f"{'='*140}\n\n")

    # With thinking
    f.write("\n" + "="*140 + "\n")
    f.write("  WITH THINKING\n")
    f.write("="*140 + "\n")

    with_thinking_pass, with_thinking_fail = extract_parsing_cases("data/hindi/qwen3_14B_results.json")

    f.write(f"\nPASSING CASES: {len(with_thinking_pass)}\n")
    f.write("-" * 140 + "\n")
    for i, case in enumerate(with_thinking_pass[:n], 1):
        f.write(f"\n[{i}] {case['table_name']}\n")
        f.write(f"    Question: {case['question']}\n")
        f.write(f"    Response(non-thinking part): {case['model_response'][:200]}...\n")

    if len(with_thinking_pass) > n:
        f.write(f"\n... and {len(with_thinking_pass) - n} more passing cases\n")

    f.write(f"\nFAILING CASES: {len(with_thinking_fail)}\n")
    f.write("-" * 140 + "\n")
    for i, case in enumerate(with_thinking_fail[:n], 1):
        f.write(f"\n[{i}] {case['table_name']}\n")
        f.write(f"    Question: {case['question']}\n")
        f.write(f"    Error: {case['error']}\n")
        f.write(f"    Response(non-thinking part): {case['model_response']}\n")

    if len(with_thinking_fail) > n:
        f.write(f"\n... and {len(with_thinking_fail) - n} more failing cases\n")

    # Without thinking
    f.write(f"\n\n" + "="*140 + "\n")
    f.write("  WITHOUT THINKING\n")
    f.write("="*140 + "\n")

    without_thinking_pass, without_thinking_fail = extract_parsing_cases("data/hindi/qwen3_14B_results_nothink.json")

    f.write(f"\nPASSING CASES: {len(without_thinking_pass)}\n")
    f.write("-" * 140 + "\n")
    for i, case in enumerate(without_thinking_pass[:n], 1):
        f.write(f"\n[{i}] {case['table_name']}\n")
        f.write(f"    Question: {case['question']}\n")
        f.write(f"    R: {case['model_response'][:200]}...\n")

    if len(without_thinking_pass) > n:
        f.write(f"\n... and {len(without_thinking_pass) - n} more passing cases\n")

    f.write(f"\nFAILING CASES: {len(without_thinking_fail)}\n")
    f.write("-" * 140 + "\n")
    for i, case in enumerate(without_thinking_fail[:n], 1):
        f.write(f"\n[{i}] {case['table_name']}\n")
        f.write(f"    Question: {case['question']}\n")
        f.write(f"    Error: {case['error']}\n")
        f.write(f"    R: {case['model_response']}\n")

    if len(without_thinking_fail) > n:
        f.write(f"\n... and {len(without_thinking_fail) - n} more failing cases\n")

    # Summary
    f.write(f"\n\n" + "="*140 + "\n")
    f.write("  SUMMARY\n")
    f.write("="*140 + "\n")
    f.write(f"\nWith Thinking:\n")
    f.write(f"  Passing:  {len(with_thinking_pass)}\n")
    f.write(f"  Failing:  {len(with_thinking_fail)}\n")
    f.write(f"  Success:  {len(with_thinking_pass)/(len(with_thinking_pass)+len(with_thinking_fail))*100:.2f}%\n")

    f.write(f"\nWithout Thinking:\n")
    f.write(f"  Passing:  {len(without_thinking_pass)}\n")
    f.write(f"  Failing:  {len(without_thinking_fail)}\n")
    f.write(f"  Success:  {len(without_thinking_pass)/(len(without_thinking_pass)+len(without_thinking_fail))*100:.2f}%\n")

    f.write(f"\n{'='*140}\n")

print(f"✓ Analysis saved to: {output_file}")


✓ Analysis saved to: data/hindi/failure/qwen3_14B_parsing_analysis.txt
